In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

# ── Parse GBSA CSV (multi-section format) ─────────────────────────────────────
def parse_gbsa(filepath):
    sections, current_section, rows, cols = {}, None, [], None
    with open(filepath) as f:
        lines = f.readlines()
    for line in lines:
        line = line.strip()
        if not line:
            if current_section and rows:
                sections[current_section] = pd.DataFrame(rows, columns=cols)
            rows, cols, current_section = [], None, None
            continue
        if line in ('GENERALIZED BORN:', 'Complex Energy Terms', 'Receptor Energy Terms',
                    'Ligand Energy Terms', 'Delta Energy Terms'):
            current_section = line.rstrip(':')
            continue
        if line.startswith('Frame #'):
            cols = line.split(',')
            continue
        if current_section and cols:
            try:
                rows.append([float(x) for x in line.split(',')])
            except:
                pass
    if current_section and rows:
        sections[current_section] = pd.DataFrame(rows, columns=cols)
    return sections

# ── Load both datasets ─────────────────────────────────────────────────────────
# Update these paths to match your file locations
FILE1 = 'ID1-ID1-FINAL_RESULTS_MMPBSA_ENERGY.csv'
FILE2 = 'ID1-ID4.csv'
LABEL1 = 'ID1-ID1'
LABEL2 = 'ID1-ID4'

sections1 = parse_gbsa(FILE1)
sections2 = parse_gbsa(FILE2)

delta1 = sections1['Delta Energy Terms']
delta2 = sections2['Delta Energy Terms']

frames1 = delta1['Frame #'].values
frames2 = delta2['Frame #'].values
total1  = delta1['TOTAL'].values
total2  = delta2['TOTAL'].values

mean1, std1 = total1.mean(), total1.std()
mean2, std2 = total2.mean(), total2.std()
window = 10
rolling1 = pd.Series(total1).rolling(window, center=True).mean().values
rolling2 = pd.Series(total2).rolling(window, center=True).mean().values
cumavg1  = pd.Series(total1).expanding().mean().values
cumavg2  = pd.Series(total2).expanding().mean().values

# ── Publication Color Palette ─────────────────────────────────────────────────
C1   = '#2166AC'   # deep blue      — ID1-ID1 main
C2   = '#D6604D'   # brick red      — ID1-ID4 main
C3   = '#1A9641'   # forest green   — mean/reference
C4   = '#762A83'   # purple         — SD/secondary
C5   = '#F4A582'   # salmon         — positive bars
C6   = '#4DAC26'   # lime green     — negative bars
GRID = '#CCCCCC'
TEXT = '#1A1A1A'
BG   = '#FFFFFF'
PANEL= '#F7F7F7'

def base_layout(title):
    return dict(
        paper_bgcolor=BG,
        plot_bgcolor=PANEL,
        font=dict(family='Arial, sans-serif', color=TEXT, size=14),
        title=dict(
            text=title,
            font=dict(size=20, color=TEXT, family='Arial, sans-serif'),
            x=0.5, xanchor='center'
        ),
        legend=dict(
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#AAAAAA',
            borderwidth=1,
            font=dict(size=12, color=TEXT)
        ),
        xaxis=dict(
            gridcolor=GRID, gridwidth=0.8,
            linecolor='#333333', linewidth=1.5,
            tickfont=dict(size=12, color=TEXT),
            title_font=dict(size=14, color=TEXT),
            mirror=True, showline=True,
            ticks='outside', ticklen=5
        ),
        yaxis=dict(
            gridcolor=GRID, gridwidth=0.8,
            linecolor='#333333', linewidth=1.5,
            tickfont=dict(size=12, color=TEXT),
            title_font=dict(size=14, color=TEXT),
            mirror=True, showline=True,
            ticks='outside', ticklen=5
        ),
        margin=dict(l=80, r=60, t=80, b=70)
    )


# ══════════════════════════════════════════════════════════════════════════════
# Plot 1 — Binding Free Energy Time Series (both datasets)
# ══════════════════════════════════════════════════════════════════════════════
fig1 = go.Figure()

for total, frames, mean, std, rolling, color, label in [
    (total1, frames1, mean1, std1, rolling1, C1, LABEL1),
    (total2, frames2, mean2, std2, rolling2, C2, LABEL2),
]:
    fig1.add_trace(go.Scatter(
        x=np.concatenate([frames, frames[::-1]]),
        y=np.concatenate([np.full_like(frames, mean + std),
                          np.full_like(frames, mean - std)[::-1]]),
        fill='toself',
        fillcolor=f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.10)',
        line=dict(color='rgba(0,0,0,0)'),
        name=f'{label} ±1 SD ({std:.2f})',
        hoverinfo='skip',
        legendgroup=label
    ))
    fig1.add_trace(go.Scatter(
        x=frames, y=total,
        mode='lines', name=f'{label} per-frame ΔG',
        line=dict(color=color, width=1.2), opacity=0.45,
        hovertemplate=f'<b>{label}</b><br>Frame: %{{x}}<br>ΔG: %{{y:.2f}} kcal/mol<extra></extra>',
        legendgroup=label
    ))
    fig1.add_trace(go.Scatter(
        x=frames, y=rolling,
        mode='lines', name=f'{label} {window}-frame mean',
        line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{label}</b><br>Frame: %{{x}}<br>Rolling mean: %{{y:.2f}} kcal/mol<extra></extra>',
        legendgroup=label
    ))
    fig1.add_hline(
        y=mean,
        line=dict(color=color, width=1.8, dash='dash'),
        annotation_text=f'<b>{label} Mean = {mean:.2f} kcal/mol</b>',
        annotation_font=dict(color=color, size=11)
    )

fig1.update_layout(
    **base_layout('Binding Free Energy over Trajectory'),
    xaxis_title='Simulation Frame',
    yaxis_title='ΔG<sub>binding</sub> (kcal/mol)',
    height=500, width=900
)
fig1.write_html('plot1_timeseries.html')
print("Saved: plot1_timeseries.html")


# ══════════════════════════════════════════════════════════════════════════════
# Plot 2 — Energy Components Bar Chart (grouped, both datasets)
# ══════════════════════════════════════════════════════════════════════════════
fig2 = go.Figure()

components  = ['VDWAALS', 'EEL', 'EGB', 'ESURF', 'GGAS', 'GSOLV']
comp_labels = ['ΔVdW', 'ΔEEL', 'ΔEGB', 'ΔESURF', 'ΔGGAS', 'ΔGSOLV']

for delta, color, label in [(delta1, C1, LABEL1), (delta2, C2, LABEL2)]:
    means = [delta[c].mean() for c in components]
    stds  = [delta[c].std()  for c in components]
    fig2.add_trace(go.Bar(
        x=comp_labels, y=means,
        error_y=dict(type='data', array=stds, visible=True,
                     color='#333333', thickness=1.5, width=5),
        marker=dict(color=color, opacity=0.85,
                    line=dict(color='#333333', width=1.2)),
        name=label,
        hovertemplate=f'<b>{label}</b><br>%{{x}}<br>Mean: %{{y:.2f}} kcal/mol<br>SD: ±%{{error_y.array:.2f}}<extra></extra>'
    ))

fig2.add_hline(y=0, line=dict(color='#333333', width=1.2))
fig2.update_layout(
    **base_layout('Mean Energy Components (ΔDelta)'),
    xaxis_title='Energy Component',
    yaxis_title='ΔΔG (kcal/mol)',
    barmode='group',
    height=500, width=800
)
fig2.write_html('plot2_components.html')
print("Saved: plot2_components.html")


# ══════════════════════════════════════════════════════════════════════════════
# Plot 3 — Convergence Check (both datasets)
# ══════════════════════════════════════════════════════════════════════════════
fig3 = go.Figure()

for total, frames, mean, std, cumavg, color, label in [
    (total1, frames1, mean1, std1, cumavg1, C1, LABEL1),
    (total2, frames2, mean2, std2, cumavg2, C2, LABEL2),
]:
    fig3.add_trace(go.Scatter(
        x=np.concatenate([frames, frames[::-1]]),
        y=np.concatenate([cumavg + std, (cumavg - std)[::-1]]),
        fill='toself',
        fillcolor=f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.10)',
        line=dict(color='rgba(0,0,0,0)'),
        name=f'{label} ±1 SD',
        hoverinfo='skip',
        legendgroup=label
    ))
    fig3.add_trace(go.Scatter(
        x=frames, y=cumavg,
        mode='lines', name=f'{label} cumulative mean',
        line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{label}</b><br>Frame: %{{x}}<br>Cumulative mean: %{{y:.2f}} kcal/mol<extra></extra>',
        legendgroup=label
    ))
    fig3.add_hline(
        y=mean,
        line=dict(color=color, width=1.8, dash='dash'),
        annotation_text=f'<b>{label} final = {mean:.2f}</b>',
        annotation_font=dict(color=color, size=11)
    )

fig3.update_layout(
    **base_layout('Convergence of Binding Free Energy'),
    xaxis_title='Simulation Frame',
    yaxis_title='Cumulative Mean ΔG (kcal/mol)',
    height=500, width=900
)
fig3.write_html('plot3_convergence.html')
print("Saved: plot3_convergence.html")


# ══════════════════════════════════════════════════════════════════════════════
# Plot 4 — Distribution Histogram + KDE (both datasets overlaid)
# ══════════════════════════════════════════════════════════════════════════════
fig4 = go.Figure()

for total, mean, std, color, label in [
    (total1, mean1, std1, C1, LABEL1),
    (total2, mean2, std2, C2, LABEL2),
]:
    fig4.add_trace(go.Histogram(
        x=total, nbinsx=20,
        marker=dict(color=color, opacity=0.45,
                    line=dict(color='#FFFFFF', width=0.8)),
        name=f'{label} distribution',
        hovertemplate=f'<b>{label}</b><br>ΔG: %{{x:.1f}} kcal/mol<br>Count: %{{y}}<extra></extra>'
    ))
    kde_x = np.linspace(total.min() - 5, total.max() + 5, 300)
    kde_y = gaussian_kde(total)(kde_x)
    bin_width = (total.max() - total.min()) / 20
    kde_scaled = kde_y * len(total) * bin_width
    fig4.add_trace(go.Scatter(
        x=kde_x, y=kde_scaled,
        mode='lines', name=f'{label} KDE',
        line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{label}</b><br>ΔG: %{{x:.2f}}<br>Density: %{{y:.2f}}<extra></extra>'
    ))
    fig4.add_vline(
        x=mean,
        line=dict(color=color, width=2, dash='dash'),
        annotation_text=f'<b>{label} Mean {mean:.2f}</b>',
        annotation_font=dict(color=color, size=11),
        annotation_position='top right'
    )

fig4.update_layout(
    **base_layout('Distribution of Binding Free Energies'),
    xaxis_title='ΔG<sub>binding</sub> (kcal/mol)',
    yaxis_title='Count',
    barmode='overlay',
    height=500, width=780,
    bargap=0.05
)
fig4.write_html('plot4_distribution.html')
print("Saved: plot4_distribution.html")


# ══════════════════════════════════════════════════════════════════════════════
# Plot 5 — VdW vs Electrostatics Scatter (both datasets)
# ══════════════════════════════════════════════════════════════════════════════
fig5 = go.Figure()

for delta, frames, color, label in [
    (delta1, frames1, C1, LABEL1),
    (delta2, frames2, C2, LABEL2),
]:
    fig5.add_trace(go.Scatter(
        x=delta['VDWAALS'].values,
        y=delta['EEL'].values,
        mode='markers',
        marker=dict(
            color=color, size=9, opacity=0.75,
            line=dict(color='#FFFFFF', width=0.5)
        ),
        name=label,
        hovertemplate=(
            f'<b>{label} — Frame %{{text}}</b><br>'
            'ΔVdW: %{x:.2f} kcal/mol<br>'
            'ΔEEL: %{y:.2f} kcal/mol<extra></extra>'
        ),
        text=[str(int(f)) for f in frames]
    ))

fig5.add_hline(y=0, line=dict(color='#AAAAAA', width=1, dash='dot'))
fig5.add_vline(x=0, line=dict(color='#AAAAAA', width=1, dash='dot'))
fig5.update_layout(
    **base_layout('VdW vs Electrostatic Contribution'),
    xaxis_title='ΔVdW (kcal/mol)',
    yaxis_title='ΔEEL (kcal/mol)',
    height=540, width=680
)
fig5.write_html('plot5_vdw_vs_eel.html')
print("Saved: plot5_vdw_vs_eel.html")

print(f"\n{'─'*50}")
print(f"{'Dataset':<12} {'Mean ΔG':>12} {'± SD':>12}")
print(f"{'─'*50}")
print(f"{LABEL1:<12} {mean1:>10.2f}  {std1:>10.2f} kcal/mol")
print(f"{LABEL2:<12} {mean2:>10.2f}  {std2:>10.2f} kcal/mol")
print(f"{'─'*50}")
